# Colab preparation

This cell gets the latest code from GitHub (clone on first run, then `git fetch` + `reset --hard origin/main`), mounts Google Drive, and installs only missing packages. It does not train or draw the figure.

Run this cell after we push updates; it prints the repo commit hash. Colab can still *display* old markdown in cells you opened earlier—trust the printed commit, or reopen the notebook from GitHub. Data: `My Drive/Foundation Model/vendor/pfdelta_task1.1`.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Repo at {head}", flush=True)
    from google.colab import drive

    drive.mount("/content/drive")

missing = []
for module, package in (
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("matplotlib", "matplotlib"),
    ("pyarrow", "pyarrow"),
    ("huggingface_hub", "huggingface_hub"),
):
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Required packages are already installed.")


# Training

This cell trains **TPGT** only (type-pair graph Transformer, independent weights per stage) on that Drive dataset. It writes checkpoints to `Foundation Model/tpgt_task11` and comparison tables to `Foundation Model/results`. It does not draw the figure.

Use a **GPU** runtime. Run the prep cell first so you have the latest `train_tpgt_task11.py` from GitHub. A finished seed (100 epochs, matching checkpoint version) is skipped if this cell is run again; obsolete Drive checkpoints (wrong version, old RPE width, or corrupt files) are ignored and training restarts from epoch 0.

Training uses feasible **N** only (same split as GENCO). **100 epochs**, batch **64**, loss **0.85** state + **0.15** physics, seeds 0–2. Each epoch prints mean train **state**, **phys**, per-stage physics, weighted **train** total, and **val_pbl**, then test **N**, **N−1**, **N−2**, and **C2I**; every **3** epochs (and at epoch 100) it also saves a Drive checkpoint and updates the comparison tables.

Optional: set `GENCO_TPGT_COMPILE=1` (or pass `--compile` / `--fast` to `%run`) for `torch.compile`; default training already uses batched type-pair attention.

Pair features are **meshed-network RPE** (full topology: \(Y_{ij}\), hop distance, effective resistance, random walks), not a spanning tree. Width/depth (32 hidden, 2 heads, 3 stages) are implementation choices, not the authors' hyperparameters.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Training from {head}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

%run train_tpgt_task11.py


# SVG figure

This cell draws the published Task 1.1 SVG (GENCO Base, CANOS-PF, GNS-S, PFNet) from the published table. It does not train.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


here = Path.cwd()
if not (here / "plot_task1_1.py").is_file():
    if on_colab():
        if not REPO.is_dir():
            subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
        os.chdir(REPO)
    elif (here / "genco" / "plot_task1_1.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("plot_task1_1.py is not in this folder.")

%run plot_task1_1.py --published
